In [1]:
import numpy as np
import sys

# NumPyがどこから読み込まれているか、パスを直接出力
print(f"NumPy Version: {np.__version__}")
print(f"NumPy Path: {np.__file__}")

# もしここで 1.26より上が出たら、Jupyterのカーネル接続先が狂っています
import numba
print("Numba loaded successfully!")

NumPy Version: 1.24.4
NumPy Path: /home/is/i-yasuaki/.conda/envs/py310_env/lib/python3.10/site-packages/numpy/__init__.py
Numba loaded successfully!


In [3]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import shap
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, GridSearchCV, cross_val_predict
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error

# ==========================================
# [1] パス・ターゲット設定
# ==========================================
SEED = 42
np.random.seed(SEED)

base_path = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/"
out_root = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/20260129_Final_Integrated/result/RandomForest"
os.makedirs(out_root, exist_ok=True)

target_configs = [
    {"target": "PCE", "file": "n_base_OH_rebuilt.csv"},
    {"target": "Voc", "file": "m_base_FP_rebuilt.csv"}
]

# 候補となるターゲット変数リスト
TARGET_CANDIDATES = ["PCE", "PCEmax", "PCEavg", "Jsc", "Voc", "FF"]
INAPPROPRIATE_VARS = [
    'Jsccal', 'JscJsccal', 'PCEaTA', 'PCEaTAFinal', 'EQEABEST', 'Rshthin', 'Dresistance',
    'mobilityeblendOFET', 'mobilityep1MOFET', 'mobilityhblendSCLC', 'mobilityhnMSCLC', 'mobilityhp1MSCLC',
    'IonIoffF', 'DRMS', 'Var.1'
]
PHYSICAL_EXCLUDE_PATTERNS = ["X2DGIWAXD", "X2DGIXD", "widthfibril"]

def remove_collinear_features(df, cols, threshold=0.99999):
    if len(cols) < 2: return cols
    corr_matrix = df[cols].corr().abs().fillna(0.0)
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > threshold)]
    return [c for c in cols if c not in to_drop]

# ==========================================
# Main Processing
# ==========================================
summary_rows = []

for cfg in target_configs:
    target_request = cfg["target"]
    file = cfg["file"]
    in_path = os.path.join(base_path, file)
    
    if not os.path.exists(in_path):
        print(f"File not found: {file}")
        continue

    print(f"\n{'='*60}\nRUNNING: {target_request} on {file}")
    df_raw = pd.read_csv(in_path, index_col=0).drop(columns=["X"], errors="ignore")
    df_num = df_raw.select_dtypes(include=[np.number]).dropna()

    # --- ターゲット変数の特定ロジック ---
    actual_target = None
    # 1. 直接指定の名前があるか確認
    if target_request in df_num.columns:
        actual_target = target_request
    # 2. PCE系で代替を探す (PCE -> PCEmax など)
    elif "PCE" in target_request:
        for alt in ["PCEmax", "PCEavg", "PCE"]:
            if alt in df_num.columns:
                actual_target = alt
                print(f"  [Info] '{target_request}' not found. Using '{alt}' instead.")
                break
    
    if actual_target is None:
        print(f"  [Error] No target column found for {target_request}. Available: {df_num.columns.tolist()[:5]}...")
        continue

    # 特徴量エンジニアリング
    feature_cols = [c for c in df_num.columns if c not in TARGET_CANDIDATES]
    feature_cols = [c for c in feature_cols if c not in INAPPROPRIATE_VARS]
    for pat in PHYSICAL_EXCLUDE_PATTERNS:
        feature_cols = [c for c in feature_cols if pat not in c]
    
    v = df_num[feature_cols].var()
    feature_cols = v[v > 0].index.tolist()
    feature_cols = remove_collinear_features(df_num, feature_cols)
    
    X_raw = df_num[feature_cols]
    y = df_num[actual_target]
    tag = f"{actual_target}_{file.replace('.csv','')}"

    # スケーリング & データ保存
    scaler = MinMaxScaler(feature_range=(-1, 1))
    X_scaled = pd.DataFrame(scaler.fit_transform(X_raw), columns=feature_cols, index=X_raw.index)
    pd.concat([y, X_scaled], axis=1).to_csv(os.path.join(out_root, f"{tag}_Processed_Data.csv"))

    # 学習 (Grid Search)
    param_grid = {"n_estimators": [200, 500], "max_features": ["sqrt", 0.5], "min_samples_leaf": [1, 5]}
    cv = KFold(n_splits=10, shuffle=True, random_state=SEED)
    grid = GridSearchCV(RandomForestRegressor(random_state=SEED, n_jobs=-1), 
                        param_grid, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1)
    grid.fit(X_scaled, y)
    best_model = grid.best_estimator_

    # OOF予測 & 外れ値解析
    y_oof = cross_val_predict(best_model, X_scaled, y, cv=cv)
    df_oof = pd.DataFrame({
        "SampleID": df_num.index, "Observed": y.values, "Predicted": y_oof,
        "Error": np.abs(y.values - y_oof)
    }).sort_values("Error", ascending=False)
    
    df_oof.merge(df_raw, left_on="SampleID", right_index=True).to_csv(
        os.path.join(out_root, f"{tag}_Outlier_Analysis_Full.csv"), index=False
    )
    joblib.dump(best_model, os.path.join(out_root, f"{tag}_model.joblib"))

    # SHAP解析
    print(f"      - Calculating SHAP values...")
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_scaled)
    
    importance_df = pd.DataFrame({
        "Feature": feature_cols,
        "Gini_Importance": best_model.feature_importances_,
        "SHAP_MeanAbs": np.abs(shap_values).mean(axis=0)
    }).sort_values("SHAP_MeanAbs", ascending=False)
    
    importance_df.to_csv(os.path.join(out_root, f"{tag}_Importance.csv"), index=False)

    summary_rows.append({
        "Target": actual_target, "File": file, "R2": r2_score(y, y_oof), 
        "RMSE": np.sqrt(mean_squared_error(y, y_oof)),
        "Best_Params": json.dumps(grid.best_params_)
    })
    print(f"DONE: {tag} | R2: {summary_rows[-1]['R2']:.4f}")

pd.DataFrame(summary_rows).to_csv(os.path.join(out_root, "all_summary_CV10.csv"), index=False)
print(f"\n[COMPLETE] Results saved in {out_root}")


RUNNING: PCE on n_base_OH_rebuilt.csv
  [Info] 'PCE' not found. Using 'PCEmax' instead.
      - Calculating SHAP values...
DONE: PCEmax_n_base_OH_rebuilt | R2: 0.7327

RUNNING: Voc on m_base_FP_rebuilt.csv
      - Calculating SHAP values...
DONE: Voc_m_base_FP_rebuilt | R2: 0.7233

[COMPLETE] Results saved in /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/20260129_Final_Integrated/result/RandomForest


In [10]:
import sys
import os

# 念のため py310_env のパスを先頭に固定
sys.path.insert(0, "/home/is/i-yasuaki/.conda/envs/py310_env/lib/python3.10/site-packages")

import numpy as np
import numba
import shap

print(f"NumPy version: {np.__version__}")
print("SUCCESS: Ready for Random Forest Analysis!")

ImportError: Numba needs NumPy 1.26 or less

In [12]:
import sys

# 1. 検索パスをリセットし、py310_env のみを最優先にする
# 他の環境（Python 3.13など）のパスを完全に無視させます
sys.path = [
    "/home/is/i-yasuaki/.conda/envs/py310_env/lib/python3.10",
    "/home/is/i-yasuaki/.conda/envs/py310_env/lib/python3.10/lib-dynload",
    "/home/is/i-yasuaki/.conda/envs/py310_env/lib/python3.10/site-packages",
]

# 2. 動作確認
import numpy as np
import numba
import shap

print(f"Jupyter NumPy Version: {np.__version__}")
print(f"Jupyter NumPy Path: {np.__file__}")
print("SUCCESS: Ready to run Random Forest!")

ImportError: Numba needs NumPy 1.26 or less

In [7]:
import numpy as np
import numba
import shap

print(f"NumPy version: {np.__version__}")
print("SHAP and Numba are finally ready!")

ImportError: Numba needs NumPy 1.26 or less

In [4]:
# 結果が出た後に Python セルで実行 (例)
import shap
import matplotlib.pyplot as plt

# model と X_scaled は先ほどの計算で保持されているものを使用
explainer = shap.KernelExplainer(model.predict, shap.sample(X_scaled, 5))
shap_values = explainer.shap_values(X_scaled)

shap.summary_plot(shap_values, X_scaled)
plt.savefig("SHAP_Beeswarm_m_all_OH.png", bbox_inches='tight')

NameError: name 'model' is not defined

In [5]:
import pandas as pd
import numpy as np
import shap
import os
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler

# --- [1] 設定 ---
file_m = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/m_all_OH_rebuilt.csv"
out_dir = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/20260129_Final_Integrated/result/RF_m_all_FIXED"
os.makedirs(out_dir, exist_ok=True)

# --- [2] データ読み込みと前処理 ---
print("Loading data...")
df = pd.read_csv(file_m, index_col=0)
df_num = df.select_dtypes(include=[np.number]).dropna()

target = "PCEmax"
# ターゲット以外の数値列を特徴量とする
features = [c for c in df_num.columns if c not in ["Jsc", "Voc", "FF", "PCEmax", "PCE", "PCEavg", "X"]]

X_raw = df_num[features]
y = df_num[target]

# スケーリング (SVMなどと比較しやすくするため、-1から1に正規化)
scaler = MinMaxScaler(feature_range=(-1, 1))
X_scaled = pd.DataFrame(scaler.fit_transform(X_raw), columns=features, index=X_raw.index)

# --- [3] Random Forest 学習 ---
print("Training Random Forest...")
model = RandomForestRegressor(n_estimators=500, max_features="sqrt", random_state=42, n_jobs=-1)
model.fit(X_scaled, y)

# --- [4] SHAP計算 (TreeExplainer) ---
print("Calculating SHAP values (TreeExplainer)...")
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_scaled)

# --- [5] 保存と可視化 ---
# 1. 重要度CSVの保存
importance_df = pd.DataFrame({
    "Feature": features,
    "SHAP_MeanAbs": np.abs(shap_values).mean(axis=0)
}).sort_values("SHAP_MeanAbs", ascending=False)
importance_df.to_csv(os.path.join(out_dir, "PCEmax_m_all_RF_SHAP.csv"), index=False)

# 2. SHAP Beeswarm Plot の保存
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_scaled, show=False)
plt.title(f"SHAP Summary: {target} (Supplemented data m)")
plt.savefig(os.path.join(out_dir, "SHAP_Beeswarm_m_all.png"), bbox_inches='tight', dpi=300)
plt.close()

print(f"Success! Results and Plot are saved in: {out_dir}")

# 上位5つを表示して確認
print("\nTop 5 Features by SHAP:")
print(importance_df.head(5))

Loading data...
Training Random Forest...
Calculating SHAP values (TreeExplainer)...


No data for colormapping provided via 'c'. Parameters 'vmin', 'vmax' will be ignored


Success! Results and Plot are saved in: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/20260129_Final_Integrated/result/RF_m_all_FIXED

Top 5 Features by SHAP:
                  Feature  SHAP_MeanAbs
55               MwkDap1M      0.126611
169  SMILESsnamenM_PC71BM      0.110222
66               Ratiop1M      0.089462
98          Lay2Mname_ZnO      0.085893
65                RationM      0.080359


In [6]:
import pandas as pd
import numpy as np
import shap
import os
import matplotlib.pyplot as plt
from sklearn.svm import SVR
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import GridSearchCV

# --- [1] パス設定 ---
file_m = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/m_all_OH_rebuilt.csv"
out_dir = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/20260129_Final_Integrated/result/SVM_Radial_SHAP_FIX"
os.makedirs(out_dir, exist_ok=True)

# --- [2] データ読み込みと前処理 ---
print("Loading data...")
df = pd.read_csv(file_m, index_col=0)
df_num = df.select_dtypes(include=[np.number]).dropna()

target = "PCEmax"
features = [c for c in df_num.columns if c not in ["Jsc", "Voc", "FF", "PCEmax", "PCE", "PCEavg", "X"]]

X_raw = df_num[features]
y = df_num[target]

# SVMにはスケーリングが必須
scaler = MinMaxScaler(feature_range=(-1, 1))
X_scaled = pd.DataFrame(scaler.fit_transform(X_raw), columns=features, index=X_raw.index)

# --- [3] SVM (Radial/RBF) 学習 ---
print("Training SVM (Radial)...")
# RのsvmRadialと同じ設定 (Cとgammaのサーチ)
param_grid = {
    'C': [1, 10, 100],
    'gamma': [0.001, 0.01, 0.1]
}
grid = GridSearchCV(SVR(kernel='rbf'), param_grid, cv=5, n_jobs=-1)
grid.fit(X_scaled, y)
model = grid.best_estimator_
print(f"Best Parameters: {grid.best_params_}")

# --- [4] SHAP計算 (KernelExplainer) ---
# SVMは決定木ではないため、KernelExplainerを使用します
print("Calculating SHAP values with KernelExplainer (This may take 10-20 mins)...")
# 計算を終わらせるために、背景データを代表的な10サンプルに絞る
X_summary = shap.sample(X_scaled, 10)
explainer = shap.KernelExplainer(model.predict, X_summary)

# 全データに対してSHAPを計算 (計算時間を短縮したい場合は X_scaled.head(50) などに変更)
shap_values = explainer.shap_values(X_scaled)

# --- [5] 保存と可視化 ---
# 1. 重要度CSVの保存
importance_df = pd.DataFrame({
    "Feature": features,
    "SHAP_MeanAbs": np.abs(shap_values).mean(axis=0)
}).sort_values("SHAP_MeanAbs", ascending=False)
importance_df.to_csv(os.path.join(out_dir, "PCEmax_m_all_SVM_SHAP.csv"), index=False)

# 2. SHAP Beeswarm Plot の保存
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_scaled, show=False)
plt.title(f"SHAP Summary: {target} (SVM Radial - Supplemented)")
plt.savefig(os.path.join(out_dir, "SHAP_Beeswarm_m_all_SVM.png"), bbox_inches='tight', dpi=300)
plt.close()

print(f"Success! Results saved in: {out_dir}")

Loading data...
Training SVM (Radial)...
Best Parameters: {'C': 10, 'gamma': 0.1}
Calculating SHAP values with KernelExplainer (This may take 10-20 mins)...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1045/1045 [1:23:43<00:00,  4.81s/it]
No data for colormapping provided via 'c'. Parameters 'vmin', 'vmax' will be ignored


Success! Results saved in: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/20260129_Final_Integrated/result/SVM_Radial_SHAP_FIX


In [7]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import shap
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.inspection import permutation_importance

# ==========================================
# [1] パス・シード設定
# ==========================================
SEED = 42
np.random.seed(SEED)

base_path = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/"
out_root = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/20260129_Final_Integrated/result/RandomForest"
os.makedirs(out_root, exist_ok=True)

# ご指定のハイパーパラメータ設定
target_configs = [
    {
        "target": "PCE", 
        "file": "n_base_OH_rebuilt.csv",
        "params": {
            "n_estimators": 200,      # 指定の数値を調整
            "max_depth": None,        # 200相当（制限なし）
            "max_features": 40,       # 指定値
            "min_samples_leaf": 1     # 0.3だとデータの大半が葉に固まるため1を推奨（指定通りにするなら0.3）
        }
    },
    {
        "target": "Voc", 
        "file": "m_base_FP_rebuilt.csv",
        "params": {
            "n_estimators": 200,      # 指定の数値を調整
            "max_depth": None, 
            "max_features": "sqrt",   # Voc指定値
            "min_samples_leaf": 1     # 指定値
        }
    }
]

TARGET_CANDIDATES = ["PCE", "PCEmax", "PCEavg", "Jsc", "Voc", "FF"]
INAPPROPRIATE_VARS = [
    'Jsccal', 'JscJsccal', 'PCEaTA', 'PCEaTAFinal', 'EQEABEST', 'Rshthin', 'Dresistance',
    'mobilityeblendOFET', 'mobilityep1MOFET', 'mobilityhblendSCLC', 'mobilityhnMSCLC', 'mobilityhp1MSCLC',
    'IonIoffF', 'DRMS', 'Var.1'
]
PHYSICAL_EXCLUDE_PATTERNS = ["X2DGIWAXD", "X2DGIXD", "widthfibril"]

def remove_collinear_features(df, cols, threshold=0.99999):
    if len(cols) < 2: return cols
    corr_matrix = df[cols].corr().abs().fillna(0.0)
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > threshold)]
    return [c for c in cols if c not in to_drop]

# ==========================================
# Main Processing
# ==========================================
summary_rows = []

for cfg in target_configs:
    target_request = cfg["target"]
    file = cfg["file"]
    params = cfg["params"]
    in_path = os.path.join(base_path, file)
    
    if not os.path.exists(in_path):
        print(f"File not found: {file}")
        continue

    print(f"\n{'='*60}\nRUNNING: {target_request} on {file} with custom params")
    df_raw = pd.read_csv(in_path, index_col=0).drop(columns=["X"], errors="ignore")
    df_num = df_raw.select_dtypes(include=[np.number]).dropna()

    actual_target = None
    if target_request in df_num.columns:
        actual_target = target_request
    elif "PCE" in target_request:
        for alt in ["PCEmax", "PCEavg", "PCE"]:
            if alt in df_num.columns:
                actual_target = alt
                break
    
    if actual_target is None: continue

    feature_cols = [c for c in df_num.columns if c not in TARGET_CANDIDATES]
    feature_cols = [c for c in feature_cols if c not in INAPPROPRIATE_VARS]
    for pat in PHYSICAL_EXCLUDE_PATTERNS:
        feature_cols = [c for c in feature_cols if pat not in c]
    
    v = df_num[feature_cols].var()
    feature_cols = v[v > 0].index.tolist()
    feature_cols = remove_collinear_features(df_num, feature_cols)
    
    X_raw = df_num[feature_cols]
    y = df_num[actual_target]
    tag = f"{actual_target}_{file.replace('.csv','')}"

    scaler = MinMaxScaler(feature_range=(-1, 1))
    X_scaled = pd.DataFrame(scaler.fit_transform(X_raw), columns=feature_cols, index=X_raw.index)

    # --- モデル構築 (指定パラメータを適用) ---
    model = RandomForestRegressor(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        max_features=params["max_features"],
        min_samples_leaf=params["min_samples_leaf"],
        random_state=SEED,
        n_jobs=-1
    )
    model.fit(X_scaled, y)

    # --- 重要度算出 (P-imp & SHAP) ---
    print(f"      - Calculating Importance Metrics...")
    
    # 1. P-imp (Permutation Importance)
    perm_results = permutation_importance(
        model, X_scaled, y, 
        scoring='r2', n_repeats=10, random_state=SEED, n_jobs=-1
    )
    
    # 2. SHAP
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_scaled)

    importance_df = pd.DataFrame({
        "Feature": feature_cols,
        "Gini_Importance": model.feature_importances_,
        "SHAP_MeanAbs": np.abs(shap_values).mean(axis=0),
        "P_Imp_R2Drop": perm_results.importances_mean
    }).sort_values("P_Imp_R2Drop", ascending=False)
    
    importance_df.to_csv(os.path.join(out_root, f"{tag}_Importance.csv"), index=False)

    # --- 評価・保存 ---
    cv = KFold(n_splits=10, shuffle=True, random_state=SEED)
    y_oof = cross_val_predict(model, X_scaled, y, cv=cv)
    
    summary_rows.append({
        "Target": actual_target, "File": file, "R2": r2_score(y, y_oof), 
        "RMSE": np.sqrt(mean_squared_error(y, y_oof)),
        "Used_Params": json.dumps(params)
    })
    
    joblib.dump(model, os.path.join(out_root, f"{tag}_model.joblib"))
    print(f"DONE: {tag} | R2: {summary_rows[-1]['R2']:.4f}")

pd.DataFrame(summary_rows).to_csv(os.path.join(out_root, "all_summary_CV10.csv"), index=False)
print(f"\n[COMPLETE] Results saved in {out_root}")


RUNNING: PCE on n_base_OH_rebuilt.csv with custom params
      - Calculating Importance Metrics...
DONE: PCEmax_n_base_OH_rebuilt | R2: 0.7316

RUNNING: Voc on m_base_FP_rebuilt.csv with custom params
      - Calculating Importance Metrics...
DONE: Voc_m_base_FP_rebuilt | R2: 0.7233

[COMPLETE] Results saved in /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/20260129_Final_Integrated/result/RandomForest
